In [11]:
from fifa.matches import wc_matches, check_match_counts, latest_year
from fifa.ingest import load_rankings, load_rankings_json
import pandas as pd
from fifa import config
import openpyxl

In [12]:
matches = wc_matches()
rankings= load_rankings()

In [13]:

rankings = pd.concat(
    [
        load_rankings(),                                  # CSV, through 2018
        load_rankings_json(config.FIFA_RANKING_JSON_2022, "2022-10-06"),      # 2022
        load_rankings_json(config.FIFA_RANKING_JSON_2026, "2026-06-05"),      # 2026
    ],
    ignore_index=True,
)

In [14]:
# Earliest match date per tournament
wc_start_date = matches.groupby("year")["date"].min()
wc_years = list(wc_start_date.index)
wc_start_date

year
1994   1994-06-17
1998   1998-06-10
2002   2002-05-31
2006   2006-06-09
2010   2010-06-11
2014   2014-06-12
2018   2018-06-14
2022   2022-11-20
2026   2026-06-11
Name: date, dtype: datetime64[us]

In [15]:
# Tag each ranking row with its year, then list the snapshot dates available per year.
rankings["year"] = rankings["rank_date"].dt.year
snapshot_dates = rankings.groupby("year")["rank_date"].unique()

# For each WC year, the latest snapshot on or before kickoff
closest_rank_date = {}
for year in wc_years:
    if year not in snapshot_dates:
        continue
    eligible = [d for d in snapshot_dates[year] if d <= wc_start_date[year]]
    if eligible:
        closest_rank_date[year] = max(eligible)
closest_rank_date

{1994: Timestamp('1994-06-14 00:00:00'),
 1998: Timestamp('1998-05-20 00:00:00'),
 2002: Timestamp('2002-05-15 00:00:00'),
 2006: Timestamp('2006-05-17 00:00:00'),
 2010: Timestamp('2010-05-26 00:00:00'),
 2014: Timestamp('2014-06-05 00:00:00'),
 2018: Timestamp('2018-06-07 00:00:00'),
 2022: Timestamp('2022-10-06 00:00:00'),
 2026: Timestamp('2026-06-05 00:00:00')}

In [16]:
# Map each year to its chosen snapshot date
rankings["snapshot_date"] = rankings["year"].map(closest_rank_date)

# Keep only the snapshot rows -> one rank per (year, country).
rank_lookup = rankings[rankings["rank_date"] == rankings["snapshot_date"]][
    ["year", "country_full", "rank"]
].copy()

In [17]:
rank_lookup[rank_lookup["country_full"].str.contains("Türkiye")]

,year,country_full,rank
57837,2022,Türkiye,45
58025,2026,Türkiye,22


In [18]:
# FIFA ranking name -> results-dataset name.
name_fixes_rankings = {
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "China PR": "China",
    "USA": "United States",
    "IR Iran": "Iran",
    "Czechia": "Czech Republic",
    "Congo DR": "DR Congo",
    "Türkiye": "Turkey",
    "Cabo Verde": "Cape Verde",
    "Cape Verde Islands": "Cape Verde",
    "Côte d'Ivoire": "Ivory Coast",
    "Serbia and Montenegro":"Serbia"

}
rank_lookup["country_full"] = rank_lookup["country_full"].replace(name_fixes_rankings)

In [19]:
# Left-merge the snapshot rank onto team_a, then team_b.
matches = matches.merge(
    rank_lookup.rename(columns={"country_full": "team_a", "rank": "team_a_rank"}),
    on=["year", "team_a"],
    how="left",
)
matches = matches.merge(
    rank_lookup.rename(columns={"country_full": "team_b", "rank": "team_b_rank"}),
    on=["year", "team_b"],
    how="left",
)

In [20]:
# Drop all entries whose rank is not available - at this point that's only Iran(1998) & Serbia(1998)
matches = matches[~(matches["team_a_rank"].isna() | matches["team_b_rank"].isna())]
matches

,date,year,stage,team_a,team_b,team_a_score,team_b_score,outcome,winner,decided_by_shootout,is_host_match,city,country,team_a_rank,team_b_rank
0,1994-06-17,1994,group,Germany,Bolivia,1,0,team_a_win,Germany,False,False,Chicago,United States,1,43
1,1994-06-17,1994,group,Spain,South Korea,2,2,draw,<NA>,False,False,Dallas,United States,5,37
2,1994-06-18,1994,group,Colombia,Romania,1,3,team_b_win,Romania,False,False,Pasadena,United States,17,7
3,1994-06-18,1994,group,Italy,Republic of Ireland,0,1,team_b_win,Republic of Ireland,False,False,East Rutherford,United States,4,14
4,1994-06-18,1994,group,United States,Switzerland,1,1,draw,<NA>,False,True,Pontiac,United States,23,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,2026-07-11,2026,knockout,Norway,England,1,2,team_b_win,England,False,False,Miami Gardens,United States,31,4
600,2026-07-14,2026,knockout,France,Spain,0,2,team_b_win,Spain,False,False,Arlington,United States,3,2
601,2026-07-15,2026,knockout,England,Argentina,1,2,team_b_win,Argentina,False,False,Atlanta,United States,4,1
602,2026-07-18,2026,knockout,France,England,4,6,team_b_win,England,False,False,Miami Gardens,United States,3,4


In [21]:
matches.columns

Index(['date', 'year', 'stage', 'team_a', 'team_b', 'team_a_score',
       'team_b_score', 'outcome', 'winner', 'decided_by_shootout',
       'is_host_match', 'city', 'country', 'team_a_rank', 'team_b_rank'],
      dtype='str')

In [23]:
matches[matches['outcome'] == 'draw']

,date,year,stage,team_a,team_b,team_a_score,team_b_score,outcome,winner,decided_by_shootout,is_host_match,city,country,team_a_rank,team_b_rank
1,1994-06-17,1994,group,Spain,South Korea,2,2,draw,<NA>,False,False,Dallas,United States,5,37
4,1994-06-18,1994,group,United States,Switzerland,1,1,draw,<NA>,False,True,Pontiac,United States,23,12
6,1994-06-19,1994,group,Cameroon,Sweden,2,2,draw,<NA>,False,False,Pasadena,United States,24,10
11,1994-06-21,1994,group,Germany,Spain,1,1,draw,<NA>,False,False,Chicago,United States,1,5
16,1994-06-23,1994,group,South Korea,Bolivia,0,0,draw,<NA>,False,False,Foxborough,United States,37,43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567,2026-06-27,2026,group,Colombia,Portugal,0,0,draw,<NA>,False,False,Miami Gardens,United States,13,5
574,2026-06-29,2026,knockout,Germany,Paraguay,1,1,draw,Paraguay,True,False,Foxborough,United States,10,41
575,2026-06-29,2026,knockout,Netherlands,Morocco,1,1,draw,Morocco,True,False,Guadalupe,Mexico,8,7
586,2026-07-03,2026,knockout,Australia,Egypt,1,1,draw,Egypt,True,False,Arlington,United States,27,29
